# Steam Game Recommender

Content-based recommendation system that helps users discover new games based on
the similarity of game descriptions, genres, and tags.

**Approach:** TF-IDF vectorization + Cosine Similarity
**Dataset:** Steam Games Dataset 2025 (94,948 games)
**Author:** ete9nal

---

## Notebook Structure
1. Data Loading & Overview
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Feature Engineering
4. Modeling & Recommendations
5. Evaluation
6. Save Artifacts for Streamlit App

---

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

## 1. Data Loading & Overview

**Мета:** завантажити датасет і зрозуміти його структуру перед будь-яким аналізом.

In [ ]:
df = pd.read_csv('../data/games_march2025_full.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Колонки з пропущеними значеннями
df.columns[df.isna().any()].tolist()

## Overview Results

- **Dataset size:** 94,948 games × 47 columns
- **Memory usage:** 471.7 MB
- **Key NaN columns for TF-IDF:** `short_description` (5,349), `detailed_description` (5,426), `name` (2)
- **Columns irrelevant for modeling:** `website`, `support_url`, `support_email`, `metacritic_url`, `notes`
- **Next step:** EDA to explore distributions, then cleaning and preprocessing

## 2. EDA

Exploratory Data Analysis of Steam dataset to get insights on data distribution.
Analyze prices, genres, review distributions, and identify a reasonable filtering threshold.

In [ ]:
df['price'].describe()

In [ ]:
# Прибираємо outliers для візуалізації (менше 0.2% ігор коштують > $100)
sns.histplot(df[df['price'] <= 100]['price'], bins=50)
plt.title('Price distribution (games under $100)')
plt.xlabel('Price, $')
plt.ylabel('Count')
plt.show()

In [ ]:
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

plt.figure(figsize=(14, 6))
sns.countplot(data=df, x='release_year', color='tab:blue')
plt.title('Games released by year')
plt.xlabel('Release year')
plt.ylabel('Games count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['num_reviews_total'].describe()

In [ ]:
# Скільки ігор мають достатньо відгуків щоб довіряти їх рейтингу
for threshold in [0, 10, 30, 50, 100]:
    print(f"num_reviews_total >= {threshold}: {(df['num_reviews_total'] >= threshold).sum()} games")

In [ ]:
# Топ-10 ігор по кількості позитивних відгуків
df.nlargest(10, 'positive')[['name', 'positive', 'negative', 'pct_pos_total']]

## EDA Results

- Most games are free or priced under $10 (median: $3.99); 184 games priced above $100 were excluded from the price plot as outliers
- Dataset contains games released between 2004 and 2025, with the largest number of releases in recent years
- Most games have very few reviews — filtering to `num_reviews_total >= 30` keeps a strong, trustworthy subset while removing low-signal entries
- **Next step:** filter the dataset and prepare text features for TF-IDF

## 3. Preprocessing & Feature Engineering

**Мета:** відфільтрувати ігри з недостатньою кількістю відгуків і підготувати текстові
ознаки (genres, tags, categories, description) для TF-IDF.

In [ ]:
# Фільтруємо ігри з малою кількістю відгуків — їх рейтинг ненадійний
df_filtered = df[df['num_reviews_total'] >= 30].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f"Before filtering: {df.shape[0]}")
print(f"After filtering: {df_filtered.shape[0]}")

In [ ]:
# 'Multi-Player' і 'Multi Player' повинні залишатись окремими токенами для TF-IDF,
# тому замінюємо пробіли в назвах категорій на підкреслення
df_filtered['categories'] = df_filtered['categories'].str.replace(' ', '_').str.replace(',', ' ')

In [ ]:
text_cols = ['genres', 'tags', 'categories', 'about_the_game']
df_filtered[text_cols] = df_filtered[text_cols].fillna("")

df_filtered['combined_text'] = (
    df_filtered['genres'] + " " +
    df_filtered['tags'] + " " +
    df_filtered['categories'] + " " +
    df_filtered['about_the_game']
)
df_filtered['combined_text'].head(3)

In [ ]:
# Прибираємо службові символи Python-репрезентацій списків/словників ([, ], {, }, ', :)
# та цифри (кількість тегів типу 'FPS': 90857 нам не потрібна як текст)
df_filtered['combined_text'] = (
    df_filtered['combined_text']
    .str.replace(r"[^\w\s]|\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df_filtered['combined_text'].head(3)

## Preprocessing & Feature Engineering Results

- Filtered dataset from 94,948 to games with `num_reviews_total >= 30` for reliable popularity signal
- Normalized multi-word category names (e.g. `Multi-Player` → `Multi_Player`) so TF-IDF treats them as single tokens
- Filled missing text in `genres`, `tags`, `categories`, `about_the_game` with empty strings
- Combined all four text fields into a single `combined_text` column and cleaned it of brackets, quotes, and digits
- **Next step:** vectorize `combined_text` with TF-IDF

## 4. Modeling & Recommendations

**Мета:** перетворити `combined_text` на числові вектори через TF-IDF і порахувати
схожість між іграми через cosine similarity.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(df_filtered['combined_text'])
X.shape

In [ ]:
similarity = cosine_similarity(X)
similarity.shape

In [ ]:
# Повна матриця важка для зберігання (36k x 36k float32 ~= 5GB),
# тому зберігаємо тільки топ-50 найсхожіших ігор для кожної гри
top_k = 50
top_indices = np.argsort(similarity, axis=1)[:, -top_k:][:, ::-1].astype(np.int32)
top_indices.shape

## Modeling Results

- TF-IDF matrix shape: `(36259, 10000)` — 36,259 games, 10,000 text features
- Cosine similarity computed across the full filtered dataset
- Reduced to top-50 most similar games per title to keep storage and lookup fast
- **Next step:** evaluate recommendation quality

## 5. Evaluation

Немає даних про реальну взаємодію користувачів з іграми, тому повноцінні метрики
типу Precision@K порахувати неможливо (нема "правильної відповіді" з реальної поведінки людей).

Натомість робимо **якісну оцінку**:
1. Перевіряємо рекомендації для відомих ігор — чи мають вони сенс
2. Порівнюємо з popularity baseline — чи наша модель дає щось краще за "просто топ ігор"
3. Дивимось на coverage — яку частку каталогу модель взагалі здатна порекомендувати

In [ ]:
def recommend(game_title, df, top_indices_matrix, top_n=5):
    matches = df[df['name'].str.contains(game_title, case=False, na=False)]
    if matches.empty:
        print(f"Game '{game_title}' not found.")
        return pd.Series(dtype='object')

    idx = matches.index[0]
    similar_idx = top_indices_matrix[idx, 1:top_n + 1]  # [0] виключаємо — це сама гра
    return df['name'].iloc[similar_idx]

recommend("Baldur's Gate", df_filtered, top_indices, top_n=5)

In [ ]:
recommend("Portal", df_filtered, top_indices, top_n=5)

In [ ]:
# Popularity baseline для порівняння — просто топ ігор без урахування контенту
popularity_baseline = df_filtered.nlargest(5, 'positive')['name']
popularity_baseline

In [ ]:
# Coverage — яка частка каталогу взагалі потрапляє в чиїсь топ-50 рекомендацій
unique_recommended = np.unique(top_indices[:, 1:])
coverage = len(unique_recommended) / df_filtered.shape[0]
print(f"Coverage: {coverage:.1%} of catalog appears in at least one recommendation list")

## Evaluation Results

- Recommendations for known titles (Baldur's Gate, Portal) return genre- and theme-consistent games,
  confirming the content-based approach captures meaningful similarity
- Unlike the popularity baseline (which returns the same handful of blockbuster titles regardless of input),
  our model adapts recommendations to the specific game queried
- Coverage is high, meaning the model doesn't collapse to recommending only a small popular subset
- **Limitation:** without real user interaction data, we cannot measure Precision/Recall against
  actual user preferences — this would be the natural next step with real interaction logs

## 6. Save Artifacts for Streamlit App

**Мета:** зберегти легкі файли (без повної similarity матриці) щоб Streamlit міг
швидко завантажувати дані і показувати рекомендації.

In [ ]:
os.makedirs('../data', exist_ok=True)

# Топ-50 індексів схожості — компактна заміна повної матриці
np.save('../data/top_indices.npy', top_indices)

# Тільки потрібні для відображення в Streamlit колонки
cols_to_save = ['name', 'genres', 'price', 'header_image', 'positive', 'negative', 'pct_pos_total']
df_filtered[cols_to_save].to_parquet('../data/games_cleaned.parquet', index=False)

print("Saved top_indices.npy and games_cleaned.parquet")